# E18 — O pedaço escolhido

O andar anterior mediu o que não se compra. Este abre o andar do enquadramento, e começa pela coisa
que nunca foi discutida: **a série**. Todo capítulo deste livro respondeu a uma pergunta sobre uma
série, e a série tem começo e fim porque alguém escolheu onde eles ficam.

A pergunta do caderno é quanto a resposta depende dessa escolha. A mesma medição é repetida em
pedaços de tamanhos diferentes, e o que se reporta não é a resposta: é a lista das respostas e a
dispersão entre elas.

Duas conclusões do livro entram no teste. A entrega do corte do primeiro capítulo --- que promete
5,14% dos dias rompendo --- e a dependência entre as duas pernas, que os capítulos da volta 4
mediram em várias vezes a independência.


In [1]:
# <- brinque com: SERIE_A, SERIE_B, PEDACOS, JANELA, CAUDA, POSTO
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, dependencia, estabilidade, graficos, promessa, volatilidade

SERIE_A = "sp500.csv"
SERIE_B = "ibov.csv"
PEDACOS = (504, 1260, 2520)
JANELA = 252
CAUDA = 0.05
POSTO = 13
CORTE = POSTO / (JANELA + 1)

retornos_a = volatilidade.retornos_log(dados.carregar_serie(SERIE_A))
retornos_b = volatilidade.retornos_log(dados.carregar_serie(SERIE_B))
comuns = retornos_a.index.intersection(retornos_b.index)
print("%s: %d dias | %s: %d comuns | a promessa do corte e %.5f"
      % (SERIE_A, retornos_a.size, SERIE_B, comuns.size, CORTE))


sp500.csv: 6718 dias | ibov.csv: 6450 comuns | a promessa do corte e 0.05138


In [2]:
# A mesma pergunta em pedacos de tres tamanhos.
linhas = []
respostas = {}
for pedaco in PEDACOS:
    valores = estabilidade.por_janela(retornos_a, lambda p: promessa.entrega(p, JANELA, CAUDA)["taxa"], pedaco)
    banda = estabilidade.banda_independente(CORTE, pedaco)
    conta = estabilidade.resumo(valores, CORTE, 2 * banda)
    respostas[pedaco] = valores
    linhas.append({"pedaco": pedaco, "pedacos": conta["pedacos"], "menor": conta["menor"],
                   "maior": conta["maior"], "dispersao": conta["dispersao"],
                   "independente": banda, "fora_da_banda": conta["fora_da_banda"]})
entrega_por_pedaco = pd.DataFrame(linhas).set_index("pedaco")
print(entrega_por_pedaco.round(4).to_string())
print()
for pedaco in PEDACOS:
    print("  pedacos de %4d dias: %s" % (pedaco, [round(v, 4) for v in respostas[pedaco]]))


        pedacos   menor   maior  dispersao  independente  fora_da_banda
pedaco                                                                 
504          13  0.0079  0.1111     0.0323        0.0098         0.5385
1260          5  0.0446  0.0704     0.0112        0.0062         0.2000
2520          2  0.0494  0.0547     0.0037        0.0044         0.0000

  pedacos de  504 dias: [np.float64(0.0437), np.float64(0.0119), np.float64(0.0317), np.float64(0.1111), np.float64(0.0079), np.float64(0.0913), np.float64(0.0437), np.float64(0.0675), np.float64(0.0278), np.float64(0.0159), np.float64(0.0317), np.float64(0.0079), np.float64(0.0595)]
  pedacos de 1260 dias: [np.float64(0.0446), np.float64(0.0704), np.float64(0.0496), np.float64(0.0446), np.float64(0.0446)]
  pedacos de 2520 dias: [np.float64(0.0547), np.float64(0.0494)]


In [3]:
# A dependencia entre as duas pernas, medida nos mesmos pedacos.
ra = dependencia.rompimentos(retornos_a).reindex(comuns).fillna(False).astype(bool)
rb = dependencia.rompimentos(retornos_b).reindex(comuns).fillna(False).astype(bool)
linhas = []
excessos = {}
for pedaco in PEDACOS:
    datas = pd.Series(comuns, index=pd.RangeIndex(comuns.size))
    valores = estabilidade.por_janela(datas, lambda fatia: dependencia.juntos(ra.reindex(fatia), rb.reindex(fatia))["excesso"], pedaco)
    excessos[pedaco] = valores
    conta = estabilidade.resumo(valores, 1.0, 0.0)
    linhas.append({"pedaco": pedaco, "pedacos": conta["pedacos"], "menor": conta["menor"],
                   "maior": conta["maior"], "dispersao": conta["dispersao"],
                   "abaixo_de_um": int((valores < 1.0).sum())})
dependencia_por_pedaco = pd.DataFrame(linhas).set_index("pedaco")
print(dependencia_por_pedaco.round(3).to_string())
print()
for pedaco in PEDACOS:
    print("  pedacos de %4d dias: %s" % (pedaco, [round(v, 2) for v in excessos[pedaco]]))


        pedacos  menor   maior  dispersao  abaixo_de_um
pedaco                                                 
504          12  2.982  11.363      2.584             0
1260          5  6.460   9.130      1.096             0
2520          2  6.902   8.855      1.381             0

  pedacos de  504 dias: [np.float64(7.05), np.float64(5.22), np.float64(7.47), np.float64(7.92), np.float64(10.98), np.float64(8.31), np.float64(8.84), np.float64(3.95), np.float64(2.98), np.float64(11.36), np.float64(8.31), np.float64(5.23)]
  pedacos de 1260 dias: [np.float64(6.86), np.float64(9.13), np.float64(6.46), np.float64(7.3), np.float64(6.55)]
  pedacos de 2520 dias: [np.float64(8.85), np.float64(6.9)]


In [4]:
# Figura 1: a entrega do corte, pedaco a pedaco, no pedaco menor.
valores = respostas[PEDACOS[0]]
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
eixo.bar(np.arange(valores.size), valores, 0.6,
         color=["#b03a2e" if abs(v - CORTE) > 2 * estabilidade.banda_independente(CORTE, PEDACOS[0]) else "#1f4e79" for v in valores])
eixo.axhline(CORTE, color="#555555", ls="--", lw=1.4, label="a promessa do corte")
eixo.axhspan(CORTE - 2 * estabilidade.banda_independente(CORTE, PEDACOS[0]),
             CORTE + 2 * estabilidade.banda_independente(CORTE, PEDACOS[0]), color="#555555", alpha=0.16,
             label="a banda da conta independente")
eixo.set_xlabel("pedaço de %d dias" % PEDACOS[0])
eixo.set_ylabel("fração dos dias que romperam o corte")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E18_o_pedaco_escolhido", 1)
plt.close(fig)
print("no pedaco menor a resposta vai de %.4f a %.4f" % (valores.min(), valores.max()))


no pedaco menor a resposta vai de 0.0079 a 0.1111


In [5]:
# Figura 2: as duas conclusoes lado a lado, no mesmo eixo de pedacos.
fig, (esq, dir_) = plt.subplots(1, 2, figsize=(9.8, 4.2), sharex=True)
pedacos = PEDACOS[0]
largura = max(respostas[pedacos].size, excessos[pedacos].size)
for eixo, valores, alvo, titulo, cor in (
        (esq, respostas[pedacos], CORTE, "a entrega do corte", "#b03a2e"),
        (dir_, excessos[pedacos], 1.0, "a dependencia entre as pernas", "#1f4e79")):
    eixo.bar(np.arange(valores.size), valores / alvo, 0.6, color=cor)
    eixo.set_xlim(-0.6, largura - 0.4)
    eixo.axhline(1.0, color="#555555", ls="--", lw=1.4)
    eixo.set_title("%s, em razão do prometido" % titulo, fontsize=10)
    eixo.grid(alpha=0.25, axis="y")
esq.set_xlabel("pedaço de %d dias" % pedacos)
dir_.set_xlabel("pedaço de %d dias --- um a menos: só os dias comuns às duas pernas" % pedacos)
esq.set_ylabel("razão entre o medido e o alvo")
fig.tight_layout()
graficos.salvar(fig, "E18_o_pedaco_escolhido", 2)
plt.close(fig)
print("razoes da entrega: %s" % [round(v / CORTE, 2) for v in respostas[pedacos]])
print("razoes da dependencia: %s" % [round(v, 2) for v in excessos[pedacos]])


razoes da entrega: [np.float64(0.85), np.float64(0.23), np.float64(0.62), np.float64(2.16), np.float64(0.15), np.float64(1.78), np.float64(0.85), np.float64(1.31), np.float64(0.54), np.float64(0.31), np.float64(0.62), np.float64(0.15), np.float64(1.16)]
razoes da dependencia: [np.float64(7.05), np.float64(5.22), np.float64(7.47), np.float64(7.92), np.float64(10.98), np.float64(8.31), np.float64(8.84), np.float64(3.95), np.float64(2.98), np.float64(11.36), np.float64(8.31), np.float64(5.23)]


## Leitura visual das figuras

Figura 1. Treze barras, uma por pedaço de 504 dias, num eixo vertical que começa em zero. A linha
tracejada é a promessa do corte e a faixa cinza é a banda que a conta independente autoriza, mais
larga que o próprio traço. Mais da metade das barras sai da faixa, e algumas passam bem acima da
borda superior dela, até quase encostar no topo da moldura; outras ficam abaixo, perto da base. A
cor é a informação mais forte da figura e a legenda não a explica: as barras são azuis ou
vermelhas, e nada na figura diz o que separa uma da outra. O eixo horizontal diz "pedaço de 504
dias" e numera os pedaços de zero a doze, sem data nenhuma: o leitor vê que a resposta varia de
quase nada a mais que o dobro da promessa, e não vê a que trecho da história pertencem os dois
extremos. Duas barras vizinhas podem estar a um ano de distância ou a dez, e a figura trata as
duas como vizinhas.

Figura 2. Dois painéis lado a lado, com a mesma pergunta respondida nos mesmos pedaços, e é a
comparação entre eles que interessa. À esquerda, a entrega do corte em razão do que foi prometido,
em barras vermelhas: elas atravessam a linha tracejada do um para cima e para baixo, e uma ou
outra sobem bem acima dela. À direita, a dependência entre as pernas na mesma razão, em barras
azuis: nenhuma barra desce até a linha, todas ficam várias vezes acima, e a escala vertical é
outra, indo até dez e mais. Duas armadilhas de leitura. A primeira salta aos olhos menos do que
deveria: os dois painéis têm a mesma largura no papel e escalas verticais diferentes, de modo que
a altura de uma barra da direita não se compara com a de uma barra da esquerda — o que se compara
é a distância de cada barra até a sua própria linha tracejada. A segunda é mais fina: o painel da
direita tem uma barra a menos que o da esquerda, e o eixo dos dois numera o pedaço em vez de
datá-lo, então a barra de mesmo número nos dois painéis não é o mesmo trecho de calendário. Quem
salta de um painel para o outro na mesma posição horizontal compara pedaços diferentes. O achado
visual é o contraste entre os dois: o corte atravessa o alvo sem cerimônia, de um lado e do outro,
enquanto a dependência nunca chega perto dele, em pedaço nenhum.


In [6]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES = {504: "dois_anos", 1260: "cinco_anos", 2520: "dez_anos"}
resultado = {
    "pedaco_dias_a": int(retornos_a.size),
    "pedaco_dias_comuns": int(comuns.size),
    "pedaco_promessa": float(CORTE),
    "pedaco_janela": int(JANELA),
    "pedaco_tamanhos": int(len(PEDACOS)),
    "pedaco_menor": int(PEDACOS[0]),
    "pedaco_maior": int(PEDACOS[-1]),
}
for pedaco in PEDACOS:
    nome = NOMES[pedaco]
    linha = entrega_por_pedaco.loc[pedaco]
    resultado["pedaco_entrega_menor_%s" % nome] = float(linha["menor"])
    resultado["pedaco_entrega_maior_%s" % nome] = float(linha["maior"])
    resultado["pedaco_entrega_dispersao_%s" % nome] = float(linha["dispersao"])
    resultado["pedaco_entrega_independente_%s" % nome] = float(linha["independente"])
    resultado["pedaco_entrega_razao_%s" % nome] = float(linha["maior"] / linha["menor"])
    resultado["pedaco_entrega_fora_%s" % nome] = float(linha["fora_da_banda"])
    resultado["pedaco_pedacos_%s" % nome] = int(linha["pedacos"])
    linha_dep = dependencia_por_pedaco.loc[pedaco]
    resultado["pedaco_dependencia_menor_%s" % nome] = float(linha_dep["menor"])
    resultado["pedaco_dependencia_maior_%s" % nome] = float(linha_dep["maior"])
    resultado["pedaco_dependencia_abaixo_%s" % nome] = int(linha_dep["abaixo_de_um"])

caminho = Path("lab/resultados/E18_o_pedaco_escolhido.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E18_o_pedaco_escolhido.json gravado | 37 grandezas
